# 01 · Preparación y calidad de los datos

**Objetivo:** caracterizar la variable de predicción y comprobar la calidad y disponibilidad de los datos utilizados en la evaluación.

Este cuaderno desarrolla la trazabilidad, la unidad de análisis, el calendario, la disponibilidad y el tratamiento de ausencias. No entrena modelos ni elimina observaciones para mejorar resultados. Las transformaciones se ejecutan en los scripts 01–04; aquí se inspeccionan y explican.

**Recorrido:** fuentes → panel → calendario → disponibilidad → cobertura → transformaciones → objetivos → extremos → límites. Las tablas se calculan al ejecutar, por lo que reflejan los archivos vigentes. Las inspecciones de toda la muestra se identifican como descriptivas; no sirven para retocar la selección congelada.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'sources.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'sources.json').is_file(), 'Abrir el cuaderno desde el repositorio o notebooks/'
REPORTS = ROOT / 'data/reports/unified'
EDA = ROOT / 'data/reports/eda'
pd.set_option('display.max_columns', 20)

In [ ]:
import numpy as np
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from forecasting.direct import build_design
from forecasting.experiment import SELECTION_CUTOFF, CALIBRATION_CUTOFF
panel = pd.read_parquet(ROOT / 'data/processed/panel_provincial_trimestral.parquet')
panel['quarter'] = pd.PeriodIndex(panel.quarter, freq='Q')
panel['province_code'] = panel.province_code.astype(str).str.zfill(2)
manifest = json.loads((ROOT / 'data/reports/data_build_manifest.json').read_text(encoding='utf-8'))
coverage = pd.read_csv(EDA / 'cobertura_variables.csv')
calendar = pd.read_csv(EDA / 'auditoria_calendario_provincial.csv', dtype={'province_code': str})

## 1. Qué representa cada observación

Una fila corresponde a un territorio y un trimestre. El objetivo es el valor medio **tasado de vivienda libre**, en euros nominales por metro cuadrado. No es el precio individual de una compraventa ni un índice corregido por las características de las viviendas. Cambios en la composición de las tasaciones pueden afectar a la media.

Se incluyen 50 provincias y Ceuta y Melilla. Una media simple de estos territorios no es una estimación ponderada del precio nacional. La amplitud temporal del panel tampoco implica que todas las variables estén observadas desde el principio.

In [ ]:
summary = pd.Series({
    'Filas de calendario': len(panel), 'Territorios': panel.province_code.nunique(),
    'Trimestres': panel.quarter.nunique(), 'Columnas': panel.shape[1],
    'Inicio': str(panel.quarter.min()), 'Fin': str(panel.quarter.max()),
    'Precios observados': int(panel.valor_tasado_m2.notna().sum()),
    'Precios ausentes': int(panel.valor_tasado_m2.isna().sum()),
}, name='Valor')
display(summary.to_frame())
display(panel[['province_code', 'province', 'quarter', 'valor_tasado_m2']].head(8))

## 2. Fuentes y trazabilidad

MIVAU proporciona tasaciones y actividad inmobiliaria; el INE aporta indicadores económicos, sociales y demográficos; el Banco de España, tipos de interés; el IGN, metadatos territoriales. `sources.json` es el inventario de descarga, no una lista de predictores ganadores. Algunas fuentes registradas son opcionales o manuales.

La tabla muestra si el archivo está en la instantánea local. El manifiesto conserva las huellas de esa instantánea: permiten detectar cambios, pero no reconstruyen las versiones que publicaban los organismos en cada fecha histórica.

In [ ]:
sources = json.loads((ROOT / 'sources.json').read_text(encoding='utf-8'))
inventory = []
for category, entries in sources.items():
    for name, item in entries.items():
        filename = item.get('filename')
        path = ROOT / 'data/raw' / filename if filename else None
        inventory.append({'fuente': name, 'grupo': category,
            'frecuencia': item.get('frequency'), 'ámbito': item.get('geography'),
            'archivo_local': bool(path and path.is_file()),
            'MB': round(path.stat().st_size / 1e6, 2) if path and path.is_file() else np.nan})
display(pd.DataFrame(inventory))
display(pd.Series({k: manifest[k] for k in ['target_imputation', 'historical_vintages_reconstructed',
                                           'availability_convention', 'revision_risk']}))

## 3. De frecuencias distintas a un panel trimestral

Los códigos provinciales se normalizan antes de unir fuentes. Los flujos mensuales se suman por trimestre y las tasas o tipos se promedian. Los stocks y los datos anuales se mantienen desde su disponibilidad supuesta; no se interpolan utilizando una publicación futura. Después se construyen intensidades por habitante y retardos.

Una suma trimestral requiere interpretar correctamente las unidades de la fuente. Que la clave sea única no demuestra por sí solo que estén presentes los tres meses originales: también deben revisarse las categorías y ausencias de los archivos de entrada. El proceso aborta si falla un adaptador requerido, en vez de publicar silenciosamente un panel incompleto.

In [ ]:
keys = ['province_code', 'quarter']
assert not panel[keys].isna().any().any()
assert not panel.duplicated(keys).any()
numeric = panel.select_dtypes(include=np.number).to_numpy(dtype=float, na_value=np.nan)
assert not np.isinf(numeric).any()
observed = panel.valor_tasado_m2.dropna()
assert observed.gt(0).all()
print('Claves únicas, sin infinitos y precios observados positivos.')
print('Estos controles estructurales no certifican por sí solos la exactitud de cada fuente.')

## 4. Calendario completo no significa precios completos

Se conserva una rejilla trimestral completa. Si falta un precio, la fila permanece con un valor ausente. Esto evita confundir «la siguiente fila disponible» con «el siguiente trimestre».

Distinguimos huecos internos de la serie y ausencias anteriores al primer precio o posteriores al último. Ninguno se convierte artificialmente en una etiqueta real de entrenamiento. El mapa siguiente representa **ausencias de precio**, no caídas del mercado.

In [ ]:
expected = pd.period_range(panel.quarter.min(), panel.quarter.max(), freq='Q')
assert all(pd.PeriodIndex(g.quarter.sort_values()).equals(expected)
           for _, g in panel.groupby('province_code'))
display(calendar.loc[calendar.n_precios_observados.lt(calendar.n_filas)])
internal = int(calendar.n_huecos_internos_precio.sum())
print(f'Huecos internos de precio: {internal}; ausencias totales: {panel.valor_tasado_m2.isna().sum()}.')
prices = panel.pivot(index='province', columns='quarter', values='valor_tasado_m2')
incomplete = prices.loc[prices.isna().any(axis=1)]
fig, ax = plt.subplots(figsize=(12, max(3, .35 * len(incomplete))))
ax.imshow(incomplete.isna().to_numpy(), aspect='auto', cmap='Greys', vmin=0, vmax=1)
ax.set_yticks(range(len(incomplete)), incomplete.index)
ticks = np.arange(0, len(prices.columns), 12)
ax.set_xticks(ticks, prices.columns[ticks].astype(str), rotation=45)
ax.set(title='Ausencias de precio en toda la muestra: negro = ausente', xlabel='Trimestre')
plt.tight_layout(); plt.show()

## 5. Ejemplo de retardo por calendario

El ejemplo de Cuenca ilustra por qué la consulta debe hacerse por fecha. Si falta un trimestre intermedio, el precio de dos o tres trimestres antes no puede presentarse como `lag1`. Del mismo modo, el objetivo anual debe corresponder exactamente a t+4, aunque la serie tenga huecos. No se utiliza el relleno de precios para crear objetivos.

In [ ]:
example = panel.loc[panel.province_code.eq('16') & panel.quarter.between('2011Q4', '2013Q2'),
    ['quarter', 'valor_tasado_m2', 'valor_tasado_m2_lag1', 'target_m2_t4']].copy()
example['fecha_del_lag1'] = example.quarter - 1
example['fecha_del_objetivo'] = example.quarter + 4
display(example)

## 6. Cuándo se considera conocida cada variable

El origen se ancla en el último valor tasado considerado publicado. Para las variables mensuales o trimestrales externas se supone un trimestre de retraso. Para las anuales, ocho trimestres después del cuarto trimestre del año de referencia. La población combina datos anuales retrasados con stocks trimestrales publicados según la convención del panel.

Estos supuestos son deliberadamente explícitos; no equivalen a un calendario real de publicaciones. Las revisiones retrospectivas siguen siendo una limitación. Los ratios se calculan después de alinear la disponibilidad de sus componentes.

In [ ]:
availability = pd.read_csv(ROOT / 'data/reports/source_availability_audit.csv')
display(availability[['component', 'variable', 'reference_frequency', 'availability_rule',
                      'first_available_quarter', 'panel_coverage_pct']])
timing = pd.DataFrame([
    {'caso': 'Indicador trimestral de 2025Q4', 'referencia': '2025Q4',
     'disponibilidad_supuesta': str(pd.Period('2025Q4') + 1)},
    {'caso': 'Indicador anual de 2020', 'referencia': '2020Q4',
     'disponibilidad_supuesta': str(pd.Period('2020Q4') + 8)},
])
display(timing)

## 7. Cobertura: qué información puede aprender un modelo

Una variable con buena cobertura actual puede no tener datos en los orígenes de entrenamiento. La comparación distingue toda la muestra, desarrollo hasta 2018Q4 y un corte inicial hasta 2010Q4. Este último es un diagnóstico de cobertura, no una máscara universal para todos los ajustes.

No se debe atribuir a una variable información que no estaba disponible. Las columnas completamente vacías se excluyen usando solo el entrenamiento de cada ajuste; las ausencias restantes se imputan dentro del modelo.

In [ ]:
variables = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'renta_neta_media_persona',
             'pib_per_capita_eur', 'hipotecas_numero_por_1000_hab',
             'viajeros_hoteles_por_1000_hab', 'pct_poblacion_18_34',
             'poblacion_extranjera', 'saldo_migratorio_total']
selected = coverage.set_index('variable').reindex(variables)
display(selected[['primera_observacion', 'cobertura_train_inicial_pct',
                  'cobertura_desarrollo_pct', 'cobertura_total_pct']])
selected[['cobertura_train_inicial_pct', 'cobertura_desarrollo_pct', 'cobertura_total_pct']].plot.barh(
    figsize=(11, 6), xlim=(0, 100), xlabel='Observaciones disponibles (%)')
plt.title('Cobertura por periodo: los tamaños de muestra son diferentes')
plt.tight_layout(); plt.show()
display(Image(filename=str(EDA / 'figuras/cobertura_anual.png')))

## 8. Unidades, ratios y escala provincial

Los totales de hipotecas o turistas están relacionados con el tamaño provincial. Por eso se consideran intensidades por 1.000 habitantes. El importe hipotecario original está en miles de euros; el ticket medio debe convertirse a euros. Estos indicadores responden preguntas diferentes: intensidad de actividad frente a volumen total.

El IPC se usa como variación interanual, no para deflactar el valor tasado. La renta es anual y no equivale a un salario mensual. Los ratios dependen de la calidad y antigüedad de ambos componentes; normalizar no elimina todos los efectos territoriales.

In [ ]:
definitions = pd.DataFrame([
    ['Turismo por 1.000 habitantes', '1.000 × viajeros trimestrales / población disponible'],
    ['Hipotecas por 1.000 habitantes', '1.000 × número de hipotecas / población disponible'],
    ['Importe medio de hipoteca (euros)', '1.000 × importe en miles de euros / número de hipotecas'],
    ['Densidad (hab./km²)', 'población disponible / superficie'],
], columns=['Indicador', 'Definición'])
display(definitions)
check = panel.dropna(subset=['viajeros_hoteles', 'poblacion', 'viajeros_hoteles_por_1000_hab'])
np.testing.assert_allclose(check.viajeros_hoteles_por_1000_hab,
                           1000 * check.viajeros_hoteles / check.poblacion)
print('La intensidad turística coincide con sus componentes disponibles.')

## 9. Etiquetas futuras y separación temporal

En un ejemplo originado en t se quiere predecir t+h. Entrenar en una fecha exige que la etiqueta t+h ya sea observable. La columna descriptiva `split` de las matrices no sustituye este control, porque el periodo admisible depende del horizonte.

Se distinguen desarrollo hasta 2018Q4, calibración posterior con objetivos hasta 2022Q4 y evaluación con objetivos desde 2023Q1. La tabla siguiente cuenta ejemplos etiquetados de desarrollo; no es el número de casos de los cuatro orígenes concretos de selección.

In [ ]:
counts = []
cutoff = pd.Period(SELECTION_CUTOFF)
for h in [1, 4, 8, 12]:
    design = build_design(panel, h)
    valid = design.quarter.le(cutoff) & design.target_quarter.le(cutoff)
    valid &= design.current_price.gt(0) & design.target_price.gt(0)
    counts.append({'horizonte': h, 'ejemplos_desarrollo': int(valid.sum()),
                   'último_origen_etiquetable': str(cutoff - h),
                   'última_fecha_objetivo': str(design.loc[valid, 'target_quarter'].max())})
display(pd.DataFrame(counts))
print('Una etiqueta ausente nunca se completa con la mediana.')

## 10. Distribuciones y extremos: investigar antes de excluir

Los extremos pueden indicar errores, pero también diferencias reales entre provincias, episodios de crisis o denominadores pequeños. El informe marca valores fuera de tres rangos intercuartílicos y no los elimina. El diagnóstico siguiente se restringe a desarrollo.

Un valor extremo del precio agregado no prueba que una vivienda individual esté mal tasada. Tampoco conviene eliminar episodios difíciles solo porque perjudican el modelo: forman parte de la capacidad predictiva que se pretende evaluar.

In [ ]:
distributions = pd.read_csv(EDA / 'distribuciones.csv')
display(distributions.loc[distributions.ambito.eq('desarrollo') &
    distributions.variable.isin(['valor_tasado_m2', 'valor_tasado_m2_yoy', 'ipc_yoy',
                                'euribor_12m', 'viajeros_hoteles_por_1000_hab'])])
dev = panel.loc[panel.quarter.le(cutoff)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(dev.valor_tasado_m2.dropna(), bins=40, color='#2878B5')
axes[0].set(title='Nivel del valor tasado en desarrollo', xlabel='€/m²', ylabel='Filas')
axes[1].hist(dev.valor_tasado_m2_yoy.dropna(), bins=40, color='#E07A3F')
axes[1].set(title='Variación interanual en desarrollo', xlabel='Variación simple (%)')
plt.tight_layout(); plt.show()

## 11. Conclusiones de calidad y enlace con el análisis

La rejilla y las etiquetas utilizan fechas trimestrales exactas; las ausencias se conservan. Las coberturas y fechas de publicación supuestas limitan qué pueden aprender los modelos. La imputación de predictores pertenece al entrenamiento, no a la construcción de los objetivos ni al conjunto completo.

Las limitaciones que permanecen son las revisiones de las fuentes, el uso de retrasos supuestos, la mezcla de coberturas y el carácter agregado del valor tasado. Los controles automáticos reducen errores de implementación, pero no convierten las fuentes en una reconstrucción perfecta del pasado.

**Siguiente cuaderno:** [02 · Análisis de variables](02_analisis_de_variables.ipynb). Estudia si las variables observadas aportan asociaciones con crecimientos futuros y si esas asociaciones son estables. No basta con que una columna esté completa para que sea útil.